# Stage 3 — 全レース再スクレイプ
history.db の全レース（約4,900件）をJRA公式から再スクレイプし、
surface/race_class/track_condition/weather/斤量/性齢/体重/オッズ等を一括補完する。

**推定所要時間:** 4,900レース × 約1.5秒 ≈ 2時間

実行前に必ずバックアップセルを実行すること。

In [ ]:
# ── セル1: Google Drive マウント & 基本設定 ──────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
# ── セル2: src/ 強制アップデート（GitHubから最新取得）─────────
import urllib.request, os, sys
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/features/engine.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
print('done')

In [ ]:
# ── セル3: history.db バックアップ ────────────────────────────
import shutil, datetime, os

db_path = f'{BASE_DIR}/data/history.db'
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
bak_path = f'{BASE_DIR}/data/history_backup_{ts}.db'

shutil.copy2(db_path, bak_path)
sz = os.path.getsize(bak_path) / 1024 / 1024
print(f'✅ バックアップ完了: {bak_path} ({sz:.1f} MB)')

In [ ]:
# ── セル4: history.db マイグレーション（新列追加・冪等）────────
import sqlite3

conn = sqlite3.connect(db_path)

# horse_history の新列
horse_new_cols = [
    ('weight_load',       'REAL'),
    ('sex',               'TEXT'),
    ('age',               'INTEGER'),
    ('body_weight',       'INTEGER'),
    ('body_weight_diff',  'INTEGER'),
    ('bracket',           'INTEGER'),
    ('corner_all',        'TEXT'),
    ('win_odds',          'REAL'),
    ('margin',            'REAL'),
    ('agari_rank',        'INTEGER'),
    ('finish_time',       'REAL'),
    ('time_diff_sec',     'REAL'),
    ('chakusa_text',      'TEXT'),
    ('class_grade',       'TEXT'),
    ('field_size',        'INTEGER'),
    ('corner_4',          'INTEGER'),
]
existing_h = {r[1] for r in conn.execute('PRAGMA table_info(horse_history)')}
for col, typ in horse_new_cols:
    if col not in existing_h:
        conn.execute(f'ALTER TABLE horse_history ADD COLUMN {col} {typ}')
        print(f'  ➕ horse_history.{col}')

# race_history の新列
race_new_cols = [
    ('track_condition', 'TEXT'),
    ('race_class',      'TEXT'),
    ('num_finishers',   'INTEGER'),
    ('weather',         'TEXT'),
    ('pace_label',      'TEXT'),
    ('race_num',        'INTEGER'),
    ('lap_times',       'TEXT'),
    ('last_3f',         'REAL'),
    ('race_name',       'TEXT'),
]
existing_r = {r[1] for r in conn.execute('PRAGMA table_info(race_history)')}
for col, typ in race_new_cols:
    if col not in existing_r:
        conn.execute(f'ALTER TABLE race_history ADD COLUMN {col} {typ}')
        print(f'  ➕ race_history.{col}')

conn.commit()
conn.close()
print('✅ マイグレーション完了')

In [ ]:
# ── セル5: 再スクレイプ対象レースIDを history.db から取得 ──────
import sqlite3

conn = sqlite3.connect(db_path)
rows = conn.execute(
    "SELECT race_id FROM race_history ORDER BY date ASC"
).fetchall()
conn.close()

race_ids = [r[0] for r in rows]
print(f'再スクレイプ対象: {len(race_ids)} レース')
print(f'期間: {race_ids[0]} 〜 {race_ids[-1]}')

In [ ]:
# ── セル6: JRAログイン ─────────────────────────────────────────
import sys
sys.path.insert(0, BASE_DIR)

import requests
from src.utils.config import JRA_BASE, HEADERS

sess = requests.Session()
sess.headers.update(HEADERS)
r = sess.get(f'{JRA_BASE}/top/top.html', timeout=10)
print(f'JRA接続: {r.status_code}')

In [ ]:
# ── セル7: 再スクレイプ本体 ────────────────────────────────────
# race_id フォーマット: YYYYMMDD_PP_RR
# 結果ページURL: pw01sde10PP RR YYYYMMDD / suffix

import time, re, sqlite3
from bs4 import BeautifulSoup
from src.scraper.jra_scraper import (
    find_r01_result, calc_suffix, parse_result_soup
)
from src.utils.config import JRA_BASE, HEADERS, PLACE_NAMES

# place_code → base URL のテンプレート
def make_base_result(place_code):
    return f'pw01sde10{place_code}'

# race_id → (date_str, place_code, race_num) に分解
def parse_race_id(race_id):
    parts = race_id.split('_')
    if len(parts) != 3:
        return None, None, None
    date_str = parts[0]   # YYYYMMDD
    place_code = parts[1] # 例 '05'
    race_num = int(parts[2])  # 例 11
    return date_str, place_code, race_num

# 日付ごとに r01 をキャッシュ（同じ会場・日は1回だけ探索）
r01_cache = {}  # (place_code, date_str) -> r01

conn = sqlite3.connect(db_path)
conn.execute('PRAGMA journal_mode=WAL')

ok = 0
skip = 0
err = 0
start_time = time.time()

TOTAL = len(race_ids)

for idx, race_id in enumerate(race_ids):
    date_str, place_code, race_num = parse_race_id(race_id)
    if not date_str:
        err += 1
        continue

    rc = PLACE_NAMES.get(place_code, '?')

    # r01 キャッシュ
    cache_key = (place_code, date_str)
    if cache_key not in r01_cache:
        base = make_base_result(place_code)
        r01 = find_r01_result(base, date_str, sess)
        r01_cache[cache_key] = r01
        if r01 is None:
            print(f'  ⚠ r01不明: {rc} {date_str}')

    r01 = r01_cache[cache_key]
    if r01 is None:
        skip += 1
        continue

    sx = calc_suffix(r01, race_num)
    base = make_base_result(place_code)
    cn = f'{base}{race_num:02d}{date_str}/{sx}'

    try:
        resp = sess.post(
            f'{JRA_BASE}/JRADB/accessS.html',
            data={'CNAME': cn}, headers=HEADERS, timeout=30
        )
        resp.encoding = 'shift_jis'
        if 'パラメータエラー' in resp.text:
            skip += 1
            time.sleep(0.3)
            continue

        soup = BeautifulSoup(resp.text, 'lxml')
        if not soup.find_all('table'):
            skip += 1
            time.sleep(0.3)
            continue

        result = parse_result_soup(soup, rc, race_num, date_str, place_code)
        if not result:
            skip += 1
            time.sleep(0.3)
            continue

        # 日付正規化
        raw_date = date_str
        norm_date = f'{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}' if len(raw_date) == 8 else raw_date

        # race_history UPDATE（全列上書き）
        conn.execute(
            """UPDATE race_history SET
                date            = ?,
                surface         = COALESCE(NULLIF(?, '不明'), surface),
                race_name       = COALESCE(NULLIF(?, ''), race_name),
                track_condition = COALESCE(?, track_condition),
                race_class      = COALESCE(NULLIF(?, ''), race_class),
                num_finishers   = COALESCE(?, num_finishers),
                weather         = COALESCE(?, weather),
                pace_label      = COALESCE(?, pace_label)
            WHERE race_id = ?""",
            (
                norm_date,
                result.get('surface'),
                result.get('race_name', ''),
                result.get('track_condition'),
                result.get('race_class', ''),
                result.get('num_finishers'),
                result.get('weather'),
                result.get('pace_label'),
                race_id,
            )
        )

        # horse_history UPDATE（新フィールドを補完）
        for h in result.get('finishers', []):
            conn.execute(
                """UPDATE horse_history SET
                    surface          = COALESCE(NULLIF(?, ''), surface),
                    weight_load      = COALESCE(?, weight_load),
                    sex              = COALESCE(NULLIF(?, ''), sex),
                    age              = COALESCE(?, age),
                    body_weight      = COALESCE(?, body_weight),
                    body_weight_diff = COALESCE(?, body_weight_diff),
                    bracket          = COALESCE(?, bracket),
                    corner_all       = COALESCE(NULLIF(?, ''), corner_all),
                    win_odds         = COALESCE(?, win_odds),
                    finish_time      = COALESCE(?, finish_time),
                    time_diff_sec    = COALESCE(?, time_diff_sec),
                    chakusa_text     = COALESCE(NULLIF(?, ''), chakusa_text),
                    margin           = COALESCE(?, margin),
                    agari_rank       = COALESCE(?, agari_rank)
                WHERE race_id = ? AND horse_num = ?""",
                (
                    h.get('surface', result.get('surface', '')),
                    h.get('weight_load'),
                    h.get('sex', ''),
                    h.get('age'),
                    h.get('body_weight'),
                    h.get('body_weight_diff'),
                    h.get('bracket'),
                    h.get('corner_all', ''),
                    h.get('win_odds'),
                    h.get('finish_time'),
                    h.get('time_diff_sec'),
                    h.get('chakusa_text', ''),
                    h.get('margin'),
                    h.get('agari_rank'),
                    race_id, h.get('num', 0),
                )
            )

        ok += 1
        time.sleep(0.8)

    except Exception as e:
        print(f'  ❌ {race_id}: {e}')
        err += 1
        time.sleep(1.0)
        continue

    # 100件ごとに途中保存 & 進捗表示
    if (idx + 1) % 100 == 0:
        conn.commit()
        elapsed = time.time() - start_time
        eta = elapsed / (idx + 1) * (TOTAL - idx - 1)
        print(
            f'  [{idx+1}/{TOTAL}] ok={ok} skip={skip} err={err}'
            f'  経過{elapsed/60:.1f}分 残り{eta/60:.1f}分'
        )

conn.commit()
conn.close()

elapsed = time.time() - start_time
print(f'\n✅ 再スクレイプ完了: ok={ok} skip={skip} err={err}  所要{elapsed/60:.1f}分')

In [ ]:
# ── セル8: 充足率検証 ─────────────────────────────────────────
import sqlite3

conn = sqlite3.connect(db_path)

print('=== race_history ===')
total_r = conn.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
for col in ['surface', 'track_condition', 'race_class', 'weather', 'num_finishers', 'race_name']:
    filled = conn.execute(
        f"SELECT COUNT(*) FROM race_history WHERE {col} IS NOT NULL AND {col} != ''"
    ).fetchone()[0]
    pct = 100 * filled / max(total_r, 1)
    mark = '✅' if pct >= 90 else '⚠' if pct >= 70 else '❌'
    print(f'  {mark} {col}: {filled}/{total_r} ({pct:.1f}%)')

print('\n=== horse_history ===')
total_h = conn.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
for col in ['surface', 'weight_load', 'sex', 'age', 'body_weight', 'bracket', 'corner_all', 'win_odds', 'finish_time']:
    filled = conn.execute(
        f"SELECT COUNT(*) FROM horse_history WHERE {col} IS NOT NULL AND {col} != ''"
    ).fetchone()[0]
    pct = 100 * filled / max(total_h, 1)
    mark = '✅' if pct >= 90 else '⚠' if pct >= 70 else '❌'
    print(f'  {mark} {col}: {filled}/{total_h} ({pct:.1f}%)')

print('\n=== surface分布（芝:ダート ≈ 55:45 が正常）===')
for tbl in ['race_history', 'horse_history']:
    rows = conn.execute(
        f'SELECT surface, COUNT(*) FROM {tbl} GROUP BY surface ORDER BY 2 DESC'
    ).fetchall()
    print(f'  {tbl}: {dict(rows)}')

print('\n=== 日付フォーマット確認（ダッシュなし件数 = 0 が正常）===')
for tbl in ['race_history', 'horse_history']:
    bad = conn.execute(
        f"SELECT COUNT(*) FROM {tbl} WHERE date NOT LIKE '____-__-__'"
    ).fetchone()[0]
    mark = '✅' if bad == 0 else '❌'
    print(f'  {mark} {tbl}.date 不正フォーマット: {bad}件')

conn.close()